# 09 · v1 / v2 training workflow

Train champion (older window) and challenger (fresher window) with the same LightGBM recipe. Both later score the same Aug 5–21 holdout.

In [1]:
import pandas as pd

from cross_model_drift.data import load_split
from cross_model_drift.features import target_vector
from cross_model_drift.metrics import quality_metrics
from cross_model_drift.models import load_model, train_lightgbm
from cross_model_drift.notebook import setup_model_session
from cross_model_drift.tracking import clearml_task, log_metrics

nb = setup_model_session()
tuned_path = nb.artifacts / "models" / "v1_lightgbm_tuned.joblib"
extra_params = {}
if tuned_path.exists():
    extra_params = load_model(tuned_path).params
extra_params

{'objective': 'binary',
 'metric': 'average_precision',
 'learning_rate': 0.030710573677773714,
 'num_leaves': 122,
 'max_depth': 10,
 'min_child_samples': 128,
 'feature_fraction': 0.5780093202212182,
 'bagging_fraction': 0.5779972601681014,
 'bagging_freq': 1,
 'lambda_l1': 3.3323645788192616e-08,
 'lambda_l2': 0.6245760287469887,
 'verbosity': -1,
 'n_jobs': -1}

In [3]:
rows = []
for version in ("v1", "v2"):
    train = load_split(version, "train", nb.config, engine=nb.engine)
    valid = load_split(version, "validation", nb.config, engine=nb.engine)
    test = load_split(version, "test", nb.config, engine=nb.engine)
    model = train_lightgbm(
        train,
        target_vector(train),
        valid,
        target_vector(valid),
        params=extra_params,
        threshold=nb.threshold,
    )
    metrics = quality_metrics(target_vector(test), model.predict_proba(test), threshold=nb.threshold)
    metrics["version"] = version
    metrics["n_train"] = len(train)
    path = model.save(nb.artifacts / "models" / f"{version}_champion_challenger.joblib")
    rows.append(metrics)
    with clearml_task(f"train_{version}", config=nb.config, task_type="training", tags=[version], init=True) as task:
        log_metrics(task, {k: v for k, v in metrics.items() if k not in ("version", "n_train")}, title=f"{version}_test")
    print(version, path)
pd.DataFrame(rows).set_index("version")

[50]	train's average_precision: 0.458255	valid's average_precision: 0.369517
[100]	train's average_precision: 0.468276	valid's average_precision: 0.375175
[150]	train's average_precision: 0.474465	valid's average_precision: 0.377794
[200]	train's average_precision: 0.478673	valid's average_precision: 0.378892
[250]	train's average_precision: 0.482236	valid's average_precision: 0.37925
[300]	train's average_precision: 0.485081	valid's average_precision: 0.379368


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML Task: created new task id=0fbc23e5bd104f399adbea17c3d45ba6
ClearML results page: http://localhost:8080/projects/23d7eaf499d345e2b9df79587a66a66c/tasks/0fbc23e5bd104f399adbea17c3d45ba6/output/log
v1 /Users/mitter/aventures/code/cross-model-drift/artifacts/models/v1_champion_challenger.joblib
[50]	train's average_precision: 0.457	valid's average_precision: 0.380846
[100]	train's average_precision: 0.467277	valid's average_precision: 0.387555
[150]	train's average_precision: 0.473689	valid's average_precision: 0.390712
[200]	train's average_precision: 0.477826	valid's average_precision: 0.392428
[250]	train's average_precision: 0.481272	valid's average_precision: 0.393477
[300]	train's average_precision: 0.484347	valid's average_precision: 0.39405


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML Task: created new task id=e126e1deface4c258b360022bf41cef7
ClearML results page: http://localhost:8080/projects/23d7eaf499d345e2b9df79587a66a66c/tasks/e126e1deface4c258b360022bf41cef7/output/log
v2 /Users/mitter/aventures/code/cross-model-drift/artifacts/models/v2_champion_challenger.joblib


,precision,recall,f1,pr_auc,roc_auc,threshold,n,n_positive,positive_rate,n_train
version,,,,,,,,,,
v1,0.699376,0.327493,0.446095,0.395006,0.852396,0.5,534680,11643,0.021776,4981627
v2,0.703690,0.330033,0.449329,0.392304,0.855949,0.5,544798,11211,0.020578,4953383
